# Profile D-MPNN Training

This notebook profiles a representative D-MPNN training step using `torch.profiler`.

- On CUDA, it records both CPU and CUDA activities.
- On Apple Silicon / MPS, PyTorch profiler support is more limited, so this notebook records CPU activity and still helps surface host-side bottlenecks and synchronization points.


In [2]:
import os
os.chdir("..")

import random
from pathlib import Path

import torch
from torch.profiler import profile, record_function, ProfilerActivity, schedule

from dmpnn.synthetic import generate_unique_graphs
from dmpnn.model import DMPNN
from dmpnn.training import DMPNNTrainer
from dmpnn.graph_utils import iter_batches, prepare_batch

In [3]:
SEED = 42
NUM_GRAPHS = 2000
BATCH_SIZE = 32
WARMUP_STEPS = 5
PROFILE_STEPS = 10
TRACE_PATH = Path("profiling") / "dmpnn_train_trace.json"

random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print(f"Using device: {device}")

Using device: mps


In [4]:
train_graphs, _ = generate_unique_graphs(NUM_GRAPHS, min_nodes=3, max_nodes=10, seed=SEED)

model = DMPNN(
    node_feat_dim=4,
    edge_feat_dim=2,
    hidden_dim=128,
    num_steps=3,
    head_hidden_size=128,
    output_size=1,
)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
trainer = DMPNNTrainer(
    model,
    optimizer=optimizer,
    loss_fn=torch.nn.MSELoss(),
    device=device,
    task_type="regression",
)

graph_batches = list(iter_batches(train_graphs, batch_size=BATCH_SIZE, shuffle=False))
batch = prepare_batch(graph_batches[0], device)

print(f"Prepared batch with {batch['num_graphs']} graphs")
print(f"X shape: {tuple(batch['X'].shape)}")
print(f"B shape: {tuple(batch['B'].shape)}")
print(f"edge_index shape: {tuple(batch['edge_index'].shape)}")

Prepared batch with 32 graphs
X shape: (206, 4)
B shape: (452, 2)
edge_index shape: (2, 452)


In [5]:
# Warm up kernels and memory allocations before profiling.
for _ in range(WARMUP_STEPS):
    trainer.train_batch(batch)

print(f"Completed {WARMUP_STEPS} warmup steps.")

Completed 5 warmup steps.


In [6]:
activities = [ProfilerActivity.CPU]
if device == "cuda":
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    schedule=schedule(wait=1, warmup=1, active=PROFILE_STEPS, repeat=1),
    record_shapes=True,
    profile_memory=True,
    with_stack=True,
) as prof:
    for step in range(PROFILE_STEPS + 2):
        with record_function("dmpnn_train_batch"):
            trainer.train_batch(batch)
        prof.step()

print("Profiling complete.")

Profiling complete.


/Users/michaelmontemurri/Desktop/Research/dmpnn-from-scratch/.venv/lib/python3.13/site-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [7]:
sort_key = "self_cuda_time_total" if device == "cuda" else "self_cpu_time_total"
print(prof.key_averages().table(sort_by=sort_key, row_limit=30))

-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  
-------------------------------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                                      dmpnn_train_batch        34.71%      14.544ms        77.76%      32.581ms       3.258ms           0 B           0 B            10  
                               Optimizer.step#Adam.step         7.96%       3.337ms        22.54%       9.445ms     944.520us           0 B           0 B            10  
                                               aten::mm         6.89%       2.889ms         6.99%       2.931ms      20.935us           0 B           

In [8]:
TRACE_PATH.parent.mkdir(parents=True, exist_ok=True)
prof.export_chrome_trace(str(TRACE_PATH))
print(f"Chrome trace written to: {TRACE_PATH}")
print("Open it in chrome://tracing or Perfetto for timeline inspection.")

Chrome trace written to: profiling/dmpnn_train_trace.json
Open it in chrome://tracing or Perfetto for timeline inspection.


## Notes

Useful places to inspect in the output:

- `dmpnn_train_batch` total time
- repeated tensor allocations in message passing and pooling
- CPU-heavy batching or indexing work
- `.item()`-style host synchronization if it shows up in the trace

If you want a fuller picture, profile a short loop over several distinct mini-batches instead of reusing one fixed batch.